# In-Class Exercise: Predicting Car Prices with Regression — SOLUTIONS

In this exercise we use F-150 listings from the car listings data to explore a core idea: **adding information reduces uncertainty**. We start by guessing a price with almost nothing to go on, then refine our guess as we learn more. At the end, we formalize this process with a linear regression model.

**Data**: Use the same `car_listings.csv` from previous assignments. Place it in this folder before running.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.nonparametric.smoothers_lowess import lowess
import statsmodels.formula.api as smf


sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# Standard cleaning pipeline (same as previous assignments)
#listings = pd.read_csv('car_listings.csv')
listings = pd.read_csv('../../2026/spring/data/car_listings.zip')

listings['time_posted']          = pd.to_datetime(listings['time_posted'], errors='coerce')
listings['year_from_time_posted'] = listings['time_posted'].dt.year
listings['year']       = pd.to_numeric(listings['year'],       errors='coerce').astype('Int64')
listings['odometer']   = pd.to_numeric(listings['odometer'],   errors='coerce').astype('Int64')
listings['post_id']    = pd.to_numeric(listings['post_id'],    errors='coerce').astype('Int64')
listings['num_images'] = pd.to_numeric(listings['num_images'], errors='coerce').astype('Int64')
listings['price']      = pd.to_numeric(listings['price'],      errors='coerce')
listings['latitude']   = pd.to_numeric(listings['latitude'],   errors='coerce')
listings['longitude']  = pd.to_numeric(listings['longitude'],  errors='coerce')

listings = listings.drop_duplicates(subset='post_id')
for col in ['make', 'model', 'location', 'title', 'fuel', 'drive',
            'transmission', 'paint', 'type', 'condition']:
    listings[col] = listings[col].str.lower()

listings['car_age'] = listings['year_from_time_posted'] - listings['year']

print(listings.shape)

In [ ]:
# Filter to F-150s with clean titles and valid numeric data in Minneapolis
f150 = (
    listings
    .query("make == 'ford' and model == 'f150'")
    .query("location == 'minneapolis'")
    .query("title == 'clean'")
    .dropna(subset=['price', 'odometer', 'car_age', 'location'])
    .query('price > 0 and odometer > 0 and car_age >= 0')
    .copy()
)
print(f'F-150 listings: {len(f150)}')

---

## Part 1: Guess the Price

Your instructor will describe a specific F-150 for sale in Minneapolis. As you learn more about it, use the `f150` data frame to refine your estimate of the listing price.

Work in the cells below — there's no single right approach. Use whatever summary statistics or visualizations seem useful.

In [ ]:
# Stage 1: All F-150 prices with the overall mean
# Discussion prompt: "Before I tell you anything — what's your best guess?"
mean_all = f150['price'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(range(len(f150)), f150['price'].values, alpha=0.25, s=8, color='steelblue')
ax.axhline(mean_all, color='red', linewidth=2, linestyle='--',
           label=f'Mean: ${mean_all:,.0f}')
ax.set_xlabel('Listing index (arbitrary order)')
ax.set_ylabel('Price ($)')
ax.set_title('All F-150 Prices')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean: ${mean_all:,.0f}   Median: ${f150["price"].median():,.0f}   SD: ${f150["price"].std():,.0f}')

In [ ]:
# Stage 2: Prices by model year
# Discussion prompt: "Now I'll tell you it's a [year]. Does that change your guess?"

# ── UPDATE each semester ──
TARGET_YEAR = 2010
# ─────────────────────────

year_means     = f150.groupby('year')['price'].mean()
mean_target_yr = year_means.get(TARGET_YEAR, float('nan'))

rng    = np.random.default_rng(42)
jitter = rng.uniform(-0.3, 0.3, size=len(f150))

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(f150['year'].astype('float64') + jitter, f150['price'].values,
           alpha=0.2, s=8, color='steelblue', label='Individual listings')
ax.scatter(year_means.index.astype('float64'), year_means.values,
           color='red', s=60, zorder=5, marker='D', label='Year mean')
ax.axhline(mean_target_yr, color='orange', linestyle=':', linewidth=2,
           label=f'{TARGET_YEAR} mean: ${mean_target_yr:,.0f}')
ax.set_xlabel('Model Year')
ax.set_ylabel('Price ($)')
ax.set_xlim([1995,2024])
ax.set_title('F-150 Prices by Model Year')
ax.legend(loc='upper left')
plt.tight_layout()
ymax = f150['price'].quantile(0.99)
ax.set_ylim(0, ymax * 1.2)

plt.show()

In [ ]:
# Stage 3: Target-year F-150s, price vs. odometer — Minneapolis vs. other cities
# Discussion prompt: "It has [odometer] miles. Now what?"

# ── UPDATE each semester ──
TARGET_ODOMETER = 192_000
ACTUAL_PRICE    = 6_500
# ─────────────────────────

f150_yr = f150[f150['year'] == TARGET_YEAR].copy()
f150_yr['group'] = f150_yr['location'].apply(
    lambda x: 'Minneapolis' if x == 'minneapolis' else 'Other cities'
)

fig, ax = plt.subplots(figsize=(8, 5))
for group, color in [('Other cities', '#cccccc'), ('Minneapolis', 'steelblue')]:
    sub = f150_yr[f150_yr['group'] == group]
    ax.scatter(sub['odometer'], sub['price'],
               color=color, alpha=0.6, s=40, label=f'{group} (n={len(sub)})')

ax.scatter([TARGET_ODOMETER], [ACTUAL_PRICE], color='red', s=200,
           zorder=10, marker='*', label=f'Our listing (${ACTUAL_PRICE:,})')

ax.set_xlabel('Odometer (miles)')
ax.set_ylabel('Price ($)')
ax.set_title(f'{TARGET_YEAR} F-150 — Price vs. Odometer')
ax.legend()
plt.tight_layout()
plt.show()

# Nearby comps: Minneapolis listings within 20% of target mileage
comps = f150_yr[
    (f150_yr['location'] == 'minneapolis') &
    (f150_yr['odometer'].between(TARGET_ODOMETER * 0.8, TARGET_ODOMETER * 1.2))
]
print(f'Minneapolis {TARGET_YEAR} F-150s within 20% of target mileage: n={len(comps)}')
if len(comps) > 0:
    print(f'  Mean: ${comps["price"].mean():,.0f}   Median: ${comps["price"].median():,.0f}')

---

*Pause here for class discussion.*

---

## Part 2: A Regression Model

What we just did by hand — filtering to a year, then to a mileage range, then to a city — is slow and hard to generalize. **Linear regression** automates it: we fit a single equation that uses all of these variables together to minimize prediction error.

We'll model `price` as a function of three features:

| Feature | Description |
|---|---|
| `car_age` | Years since the model year (at time of listing) |
| `log_odometer` | Natural log of odometer reading — compresses the long right tail of mileage |
| `in_minneapolis` | 1 if the listing is from Minneapolis, 0 otherwise |

In [ ]:
# Let's start be going back to a less restrictive data set. I'm going 
# to reuse the name
f150 = (
    listings
    .query("make == 'ford' and model == 'f150'")
    .query("title == 'clean'")
    .dropna(subset=['price', 'odometer', 'car_age', 'location'])
    .query('price > 0 and odometer > 0 and car_age >= 0')
    .copy()
)
print(f'F-150 listings: {len(f150)}')

### 2a. Feature Engineering

`car_age` already exists from the cleaning step. Create the other two features.

In [ ]:
# Natural log of odometer
f150['log_odometer'] = np.log(f150['odometer'])

# Indicator for Minneapolis listings
f150['in_minneapolis'] = (f150['location'] == 'minneapolis').astype(int)

f150[['car_age', 'log_odometer', 'in_minneapolis', 'price']].describe()

### 2b. Fitting the Model

`statsmodels.formula.api` lets us specify models with R-style formula strings. The formula `'price ~ car_age + log_odometer + in_minneapolis'` means: model `price` as a linear function of those three predictors.

Hint: call `smf.ols(formula, data=f150).fit()` and save the result as `results`.

In [ ]:
results = smf.ols('price ~ car_age + log_odometer + in_minneapolis', data=f150).fit()

In [ ]:
results.summary()

### 2c. Interpreting the Results

The **coef** column in the summary gives the estimated coefficients. Each coefficient tells you how much the predicted price changes when that variable increases by 1, holding everything else fixed.

Extract the individual coefficients below and answer each question.

**Q1. `car_age` coefficient**

Extract the coefficient and complete the print statement.

Hint: call `.params['car_age']` on `results`.

In [ ]:
coef_age = results.params['car_age']
print(f'Each additional year of age is associated with a ${coef_age:,.0f} change in price.')

*Does the sign (positive or negative) make intuitive sense? Why or why not?*

**Answer**: The sign is negative — older trucks are worth less. This makes sense: depreciation erodes value over time as vehicles accumulate wear, go out of style, and fall behind on features. The model says each additional year of age is associated with roughly that many dollars off the price, holding mileage and location constant.

**Q2. `log_odometer` coefficient**

Extract the coefficient for `log_odometer`.

Hint: call `.params['log_odometer']` on `results`.

In [ ]:
coef_odo = results.params['log_odometer']
print(f'log_odometer coefficient: {coef_odo:,.0f}')

*Because we used the natural log of odometer, a 1-unit increase in `log_odometer` means multiplying mileage by e (roughly 2.72x) — e.g., going from 50k to 136k miles. In plain English, what does this coefficient tell you?*

**Answer**: The coefficient is negative, confirming that more miles means lower price. The log scale captures the fact that the first 50,000 miles of depreciation matters more than going from 200,000 to 250,000 miles — the relationship compresses the extremes. A truck with 2.72x as many miles as another (holding age and location fixed) is predicted to sell for roughly `coef_odo` dollars less.

--- 

It can be a real challenge to work with logged data. One cool thing about regression models is it's very easy to do some scenario exploration, by making some dummy data and getting predictions.

In [ ]:
# Simple scenario table: same truck, different mileage
# Assumptions held constant:
# - 2010 F-150
# - Minneapolis listing
# - Posted in 2026 (so car_age = 16)

miles = [50_000, 100_000, 150_000, 200_000, 250_000]
car_age = 2026 - 2010

scenarios = pd.DataFrame({
    'car_age': [car_age] * len(miles),
    'log_odometer': np.log(miles),
    'in_minneapolis': [1] * len(miles),
    'odometer_miles': miles
})

scenarios['estimated_price'] = results.predict(scenarios)
scenarios['diff_from_prev'] = scenarios['estimated_price'].diff()

display(
    scenarios[['odometer_miles', 'log_odometer', 'estimated_price', 'diff_from_prev']]
    .assign(
        log_odometer=lambda d: d['log_odometer'].round(3),
        estimated_price=lambda d: d['estimated_price'].round(0).astype(int),
        diff_from_prev=lambda d: d['diff_from_prev'].round(0).astype('Int64')
    )
    .rename(columns={
        'odometer_miles': 'Miles',
        'log_odometer': 'log(Miles)',
        'estimated_price': 'Estimated Price ($)',
        'diff_from_prev': 'Diff from previous row ($)'
    })
)

**Q3. `in_minneapolis` coefficient**

Extract the coefficient for `in_minneapolis`.

Hint: call `.params['in_minneapolis']` on `results`.

In [ ]:
coef_mpls = results.params['in_minneapolis']
print(f'Minneapolis price effect: ${coef_mpls:,.0f}')

*Compared to an otherwise identical listing in another city in the dataset, what does this coefficient say about F-150 prices in Minneapolis?*

**Answer**: This is a **dummy variable** coefficient — it represents the average price premium (or discount) for Minneapolis listings after controlling for age and mileage. If positive, Minneapolis F-150s tend to list higher than comparable trucks elsewhere; if negative, they're cheaper. The magnitude tells you how much of the price difference is attributable to location alone, not to the age or condition of the truck.

**Q4. R-squared**

Look at the `R-squared` value in `results.summary()`. What fraction of the variation in F-150 prices does this model explain? Does that seem like a lot or a little, given what else might affect used car prices?

**Answer**: R² for this model is typically in the range of 0.55–0.70, meaning the model explains roughly 55–70% of the variation in F-150 prices. That's reasonably strong for a three-variable model. The remaining variation comes from factors not captured here: trim level, condition, number of owners, accident history, and seller motivation. The model gives a principled starting point, but leaves meaningful uncertainty on any specific listing.

### 2d. Making a Prediction

Your instructor described a specific F-150. Fill in its features and use the model to predict its price.

Hint: build a one-row DataFrame with the right column names, then call `.predict(target)` on `results`.

In [ ]:
# ── UPDATE each semester ──────────────────────────────────────────────────────
# 2010 F-150, listed 2026, 192k miles, Minneapolis
listing_year      = 2010
listing_post_year = 2026
listing_odometer  = 192_000
actual_price      = 6_500
# ─────────────────────────────────────────────────────────────────────────────

target = pd.DataFrame({
    'car_age':        [listing_post_year - listing_year],
    'log_odometer':   [np.log(listing_odometer)],
    'in_minneapolis': [1],
})

predicted_price = results.predict(target)

print(f'Predicted price:  ${predicted_price.iloc[0]:,.0f}')
print(f'Actual price:     ${actual_price:,.0f}')
print(f'Residual:         ${actual_price - predicted_price.iloc[0]:,.0f}  (actual minus predicted)')

*Your instructor will now reveal the actual listing price. How does the model's prediction compare? Would you call this a good deal, a bad deal, or about what the model expects?*

**Answer**: The listing price is **$6,500**. The model predicts a higher price for a 16-year-old F-150 with 192,000 miles in Minneapolis, so the residual is negative — the truck is listed *below* the model's expectation. By the model's logic, this is a good deal. Of course, the model can't see condition, trim, or the missing topper — so "good deal" is always provisional.

--- 

## Part 3: Code for slides

The below code was used for the lecture slides.

In [ ]:
for_example = (f150
               .query("odometer > 100_000 and odometer < 150_000")
               .query("price > 1000 and price < 50_000")
               .copy())

In [ ]:
# Scatterplot: odometer vs. price with linear fit
plt.figure(figsize=(9, 6))
sns.regplot(
    data=for_example,
    x='odometer',
    y='price',
    scatter_kws={'alpha': 0.25, 's': 20, 'color': 'steelblue'},
    line_kws={'color': 'crimson', 'linewidth': 2},
    ci=False
)

plt.xlabel('Odometer (miles)')
plt.ylabel('Price ($)')
plt.title('F-150 Price vs. Odometer (with linear fit)')
plt.ylim([0,40_000])
plt.tight_layout()
plt.show()

In [ ]:
# Normalize by 1,000 for easier interpretation
f150_model = for_example.copy()
f150_model['price_k'] = f150_model['price'] / 1000
f150_model['odometer_k'] = f150_model['odometer'] / 1000

# Fit model
results_k = smf.ols('price_k ~ odometer_k', data=f150_model).fit()


In [ ]:
results_k.params

In [ ]:
coef_odo_k = results_k.params['odometer_k']

print(
    f"Odometer coefficient: {coef_odo_k*1000:.4f} ($ per 1,000 miles)\n"
    f"Interpretation: each additional 1,000 miles is associated with a\n"
    f"${abs(coef_odo_k)*1000:,.0f} decrease in price, holding other variables constant."
)

In [ ]:
# Plot each graphic separately (not in a panel)

# 1) Scatter with fitted line + highlighted residuals
fig1, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    f150_model_plot['odometer_k'],
    f150_model_plot['price_k'],
    alpha=0.25,
    s=15,
    color='steelblue',
    label='Listings'
)

x_range = np.linspace(
    f150_model_plot['odometer_k'].min(),
    f150_model_plot['odometer_k'].max(),
    200
)
y_fit = intercept + coef_odo_k * x_range
ax.plot(x_range, y_fit, color='crimson', linewidth=2, label='Fitted line')

for ex, color, label in [
    (pos_ex, 'green', 'Positive residual'),
    (neg_ex, 'orange', 'Negative residual'),
]:
    pred_k = intercept + coef_odo_k * ex['odometer_k']
    ax.scatter(ex['odometer_k'], ex['price_k'], color=color, s=120, zorder=5, label=label)
    ax.annotate(
        '',
        xy=(ex['odometer_k'], ex['price_k']),
        xytext=(ex['odometer_k'], pred_k),
        arrowprops=dict(arrowstyle='->', color=color, lw=2)
    )

ax.set_xlabel('Odometer (1,000 miles)')
ax.set_ylabel('Price ($1,000s)')
ax.set_title('Price vs. Odometer — Residuals Highlighted')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# 2) Residual plot in its own figure
fig2, ax2 = plt.subplots(figsize=(8, 5))

ax2.scatter(
    f150_model_plot['predicted_price_k'],
    f150_model_plot['residual_k'],
    alpha=0.25,
    s=15,
    color='steelblue'
)
ax2.axhline(0, color='crimson', linewidth=2, linestyle='--')

for ex, color, label in [
    (pos_ex, 'green', 'Positive residual'),
    (neg_ex, 'orange', 'Negative residual'),
]:
    ax2.scatter(
        ex['predicted_price_k'],
        ex['residual_k'],
        color=color,
        s=120,
        zorder=5,
        label=label
    )

ax2.set_xlabel('Fitted Price ($1,000s)')
ax2.set_ylabel('Residual ($1,000s)')
ax2.set_title('Residual Plot')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
for_example = (f150
               .query("odometer > 0 and odometer < 500_000")
               .query("price > 1000 and price < 50_000")
               .copy())

In [ ]:
# Scatterplot: odometer vs. price with linear fit
plt.figure(figsize=(9, 6))
sns.regplot(
    data=for_example,
    x='odometer',
    y='price',
    scatter_kws={'alpha': 0.25, 's': 20, 'color': 'steelblue'},
    line_kws={'color': 'crimson', 'linewidth': 2},
    ci=False
)

plt.xlabel('Odometer (miles)')
plt.ylabel('Price ($)')
plt.title('F-150 Price vs. Odometer (with linear fit)')
plt.ylim([0,50_000])
plt.tight_layout()
plt.show()

In [ ]:
# Fit price ~ odometer on for_example
results_odo = smf.ols('price ~ odometer', data=for_example).fit()

# Compute fitted values and residuals
for_example_plot = for_example.copy()
for_example_plot['fitted'] = results_odo.fittedvalues
for_example_plot['residual'] = results_odo.resid

# Residual plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(for_example_plot['fitted'], for_example_plot['residual'],
           alpha=0.25, s=15, color='steelblue')
ax.axhline(0, color='crimson', linewidth=2, linestyle='--')
ax.set_xlabel('Fitted Price ($)')
ax.set_ylabel('Residual ($)')
ax.set_title('Residual Plot: price ~ odometer')
smoothed = lowess(for_example_plot['residual'], for_example_plot['fitted'], frac=0.3)
ax.plot(smoothed[:, 0], smoothed[:, 1], color='orange', linewidth=2, label='Lowess smooth')
ax.legend()
plt.tight_layout()
plt.show()



In [ ]:
# Fit price ~ odometer on for_example
results_odo = smf.ols('price ~ log_odometer', data=for_example).fit()

# Compute fitted values and residuals
for_example_plot = for_example.copy()
for_example_plot['fitted'] = results_odo.fittedvalues
for_example_plot['residual'] = results_odo.resid

# Residual plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(for_example_plot['fitted'], for_example_plot['residual'],
           alpha=0.25, s=15, color='steelblue')
ax.axhline(0, color='crimson', linewidth=2, linestyle='--')
ax.set_xlabel('Fitted Price ($)')
ax.set_ylabel('Residual ($)')
ax.set_title('Residual Plot: price ~ log(odometer)')
smoothed = lowess(for_example_plot['residual'], for_example_plot['fitted'], frac=0.3)
ax.plot(smoothed[:, 0], smoothed[:, 1], color='orange', linewidth=2, label='Lowess smooth')
ax.legend()
plt.tight_layout()
plt.show()
